# Project 15 — Final comparison
Run `final_data.ipynb` first. All interpretation is in `final_checkpoint.pdf`.

In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"

import platform
import random
import re
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

OUTPUT_DIR = Path.cwd()
RANDOM_SEED = 15
MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
MAX_TOKENS = 128
BATCH_SIZE = 8
VALIDATION_SHARE = 0.20
VALIDATION_FPR_LIMIT = 0.05
NEAR_DUPLICATE_SIMILARITY = 0.90
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))

DATASET_SOURCES = {
    "deepset": {
        "repo": "deepset/prompt-injections",
        "revision": "4f61ecb038e9c3fb77e21034b22511b523772cdd",
        "files": {
            "train": "data/train-00000-of-00001-9564e8b05b4757ab.parquet",
            "test": "data/test-00000-of-00001-701d16158af87368.parquet",
        },
    },
    "safeguard": {
        "repo": "xTRam1/safe-guard-prompt-injection",
        "revision": "a3a877d608f37b7d20d9945671902df895ecdb46",
        "files": {
            "train": "data/train-00000-of-00001.parquet",
            "test": "data/test-00000-of-00001.parquet",
        },
    },
}
SAFEGUARD_SAMPLE_SIZES = {"train": 800, "test": 200}

hardware = platform.machine() + f" CPU; {os.cpu_count()} logical cores"
if DEVICE == "cuda":
    hardware = torch.cuda.get_device_name(0)
print("Python:", platform.python_version())
print("Hardware:", hardware)

Python: 3.13.16
Hardware: Tesla T4


In [2]:
dataset_frames = []

for dataset_name, source in DATASET_SOURCES.items():
    for publisher_split, filename in source["files"].items():
        download_path = hf_hub_download(
            source["repo"],
            filename,
            repo_type="dataset",
            revision=source["revision"], token=False,
        )
        frame = pd.read_parquet(download_path)
        frame["source_index"] = np.arange(len(frame))

        if dataset_name == "safeguard":
            sample_indices, _ = train_test_split(
                np.arange(len(frame)),
                train_size=SAFEGUARD_SAMPLE_SIZES[publisher_split],
                stratify=frame["label"],
                random_state=RANDOM_SEED,
            )
            frame = frame.iloc[np.sort(sample_indices)].copy()

        frame["dataset"] = dataset_name
        frame["publisher_split"] = publisher_split
        frame["row_id"] = [
            f"{dataset_name}:{publisher_split}:{index}"
            for index in frame["source_index"]
        ]
        dataset_frames.append(frame)

loaded_prompts = pd.concat(dataset_frames, ignore_index=True)
print("Loaded rows:", len(loaded_prompts))


def normalize_text(text):
    normalized = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", normalized).strip()


loaded_prompts["normalized"] = loaded_prompts["text"].map(normalize_text)
assert loaded_prompts["label"].isin([0, 1]).all()
assert loaded_prompts["text"].notna().all()
assert loaded_prompts["normalized"].ne("").all()

audit_records = []
for dataset_name, dataset in loaded_prompts.groupby("dataset"):
    audit_records.append(
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "source_columns": 2,
            "negative": int(dataset["label"].eq(0).sum()),
            "positive": int(dataset["label"].eq(1).sum()),
            "positive_pct": 100 * dataset["label"].eq(1).mean(),
            "missing_target": int(dataset["label"].isna().sum()),
            "missing_text": int(dataset["text"].isna().sum()),
            "blank_text": int(dataset["normalized"].eq("").sum()),
            "exact_excess_text": int(dataset.duplicated("text").sum()),
            "exact_excess_text_label": int(dataset.duplicated(["text", "label"]).sum()),
            "normalized_excess": int(dataset.duplicated("normalized").sum()),
        }
    )

audit_results = pd.DataFrame(audit_records)
display(audit_results)
print(
    "Entity and timestamp checks: unavailable; source files contain only text and label."
)

Loaded rows: 1662


,dataset,rows,source_columns,negative,positive,positive_pct,missing_target,missing_text,blank_text,exact_excess_text,exact_excess_text_label,normalized_excess
0,deepset,662,2,399,263,39.728097,0,0,0,0,0,0
1,safeguard,1000,2,695,305,30.500000,0,0,0,3,3,3


Entity and timestamp checks: unavailable; source files contain only text and label.


In [3]:
duplicate_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    dtype=np.float32,
)
duplicate_features = duplicate_vectorizer.fit_transform(loaded_prompts["normalized"])
text_similarities = cosine_similarity(duplicate_features)

# The upper triangle counts each related pair once, without self-pairs.
related_pairs = np.triu(text_similarities >= NEAR_DUPLICATE_SIMILARITY, k=1)
left_indices, right_indices = np.where(related_pairs)
group_parents = np.arange(len(loaded_prompts))


def find_group_root(row_index):
    while group_parents[row_index] != row_index:
        group_parents[row_index] = group_parents[group_parents[row_index]]
        row_index = group_parents[row_index]
    return row_index


def merge_groups(first_index, second_index):
    first_root = find_group_root(first_index)
    second_root = find_group_root(second_index)
    if first_root != second_root:
        group_parents[second_root] = first_root


for first_index, second_index in zip(left_indices, right_indices):
    merge_groups(first_index, second_index)

for matching_indices in loaded_prompts.groupby("normalized").indices.values():
    for row_index in matching_indices[1:]:
        merge_groups(matching_indices[0], row_index)

loaded_prompts["group"] = [
    find_group_root(row_index) for row_index in range(len(loaded_prompts))
]

near_duplicate_records = []
for dataset_name in DATASET_SOURCES:
    both_in_dataset = (
        loaded_prompts["dataset"].iloc[left_indices].to_numpy() == dataset_name
    ) & (loaded_prompts["dataset"].iloc[right_indices].to_numpy() == dataset_name)
    dataset_pairs = list(
        zip(
            left_indices[both_in_dataset],
            right_indices[both_in_dataset],
        )
    )
    affected_rows = set()
    nonexact_pair_count = 0
    for first_index, second_index in dataset_pairs:
        affected_rows.update([first_index, second_index])
        if (
            loaded_prompts["text"].iloc[first_index]
            != loaded_prompts["text"].iloc[second_index]
        ):
            nonexact_pair_count += 1

    near_duplicate_records.append(
        {
            "dataset": dataset_name,
            "near_pairs": len(dataset_pairs),
            "nonexact_near_pairs": nonexact_pair_count,
            "rows_in_near_pairs": len(affected_rows),
        }
    )

near_duplicate_audit = pd.DataFrame(near_duplicate_records)
display(near_duplicate_audit)

,dataset,near_pairs,nonexact_near_pairs,rows_in_near_pairs
0,deepset,28,28,49
1,safeguard,17,14,24


In [4]:
loaded_prompts["split"] = loaded_prompts["publisher_split"]
for dataset_name in DATASET_SOURCES:
    publisher_training = loaded_prompts["dataset"].eq(dataset_name) & loaded_prompts[
        "publisher_split"
    ].eq("train")
    training_indices, validation_indices = train_test_split(
        loaded_prompts.index[publisher_training].to_numpy(),
        test_size=VALIDATION_SHARE,
        stratify=loaded_prompts.loc[publisher_training, "label"],
        random_state=RANDOM_SEED,
    )
    loaded_prompts.loc[training_indices, "split"] = "train"
    loaded_prompts.loc[validation_indices, "split"] = "validation"

# Preserve evaluation rows: test takes priority over validation, then training.
split_priority = {"train": 0, "validation": 1, "test": 2}
loaded_prompts["priority"] = loaded_prompts["split"].map(split_priority)
highest_group_priority = loaded_prompts.groupby("group")["priority"].transform("max")
survives_boundary_check = loaded_prompts["priority"].eq(highest_group_priority)

surviving_prompts = loaded_prompts.loc[survives_boundary_check].sort_values(
    ["dataset", "publisher_split", "source_index"]
)
retained_indices = surviving_prompts.drop_duplicates("normalized").index
removed_prompts = loaded_prompts.loc[
    ~loaded_prompts.index.isin(retained_indices)
].copy()
retained_prompts = (
    loaded_prompts.loc[retained_indices].sort_index().reset_index(drop=True)
)

removal_counts = (
    removed_prompts.groupby(["dataset", "split"]).size().rename("removed").reset_index()
)
display(removal_counts)

training_prompts = retained_prompts[retained_prompts["split"].eq("train")]
leakage_counts = {}
for evaluation_split in ["test", "validation"]:
    evaluation_prompts = retained_prompts[
        retained_prompts["split"].eq(evaluation_split)
    ]
    for field, check_name in [
        ("text", "exact"),
        ("normalized", "normalized"),
        ("group", "near_groups"),
    ]:
        overlapping_rows = training_prompts[field].isin(set(evaluation_prompts[field]))
        leakage_counts[f"train_{check_name}_{evaluation_split}"] = int(
            overlapping_rows.sum()
        )

assert not any(leakage_counts.values())
test_groups = set(retained_prompts.loc[retained_prompts["split"].eq("test"), "group"])
validation_groups = set(
    retained_prompts.loc[retained_prompts["split"].eq("validation"), "group"]
)
assert test_groups.isdisjoint(validation_groups)

split_counts = (
    retained_prompts.groupby(["dataset", "split"])
    .agg(
        rows=("label", "size"),
        positive=("label", "sum"),
    )
    .reset_index()
)
split_counts["negative"] = split_counts["rows"] - split_counts["positive"]
split_counts["positive_pct"] = 100 * split_counts["positive"] / split_counts["rows"]

retained_prompts[["row_id", "dataset", "split", "label"]].to_csv(
    "split.csv", index=False
)
display(split_counts)
print("Training overlap counts:", leakage_counts)

,dataset,split,removed
0,deepset,train,11
1,deepset,validation,1
2,safeguard,test,1
3,safeguard,train,9


,dataset,split,rows,positive,negative,positive_pct
0,deepset,test,116,60,56,51.724138
1,deepset,train,425,158,267,37.176471
2,deepset,validation,109,40,69,36.697248
3,safeguard,test,199,63,136,31.658291
4,safeguard,train,631,192,439,30.427892
5,safeguard,validation,160,48,112,30.000000


Training overlap counts: {'train_exact_test': 0, 'train_normalized_test': 0, 'train_near_groups_test': 0, 'train_exact_validation': 0, 'train_normalized_validation': 0, 'train_near_groups_validation': 0}


In [5]:
from peft import LoraConfig, get_peft_model
from sklearn.decomposition import PCA
import gc
import zlib

SEEDS = [15, 16, 17]
perturbations = pd.read_csv("perturbations.csv")
assert not perturbations.duplicated(["row_id", "column"]).any()
concept_pairs = pd.read_csv("concept_pairs.csv")

if Path("midsemester_scores.csv").exists():
    old_scores = pd.read_csv("midsemester_scores.csv")
else:
    old_scores = pd.read_csv("scores.csv")
    assert "test_set" in old_scores
    old_scores.to_csv("midsemester_scores.csv", index=False)

training_mask = retained_prompts.split.eq("train").to_numpy()
validation_mask = retained_prompts.split.eq("validation").to_numpy()
test_mask = retained_prompts.split.eq("test").to_numpy()
labels = retained_prompts.label.to_numpy()

# Training examples and labels are identical across all adaptation seeds.
print("Training / validation / test:", training_mask.sum(), validation_mask.sum(), test_mask.sum())
print("Representation drift decision: yes >= 0.05; no <= 0.01; otherwise undecided.")


Training / validation / test: 1056 269 315
Representation drift decision: yes >= 0.05; no <= 0.01; otherwise undecided.


In [6]:
model_load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, use_fast=False, token=False
)
encoder = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, token=False).to(DEVICE)
encoder.requires_grad_(False)
encoder.eval()
model_load_seconds = time.perf_counter() - model_load_start

assert not any(parameter.requires_grad for parameter in encoder.parameters())
print(
    "Frozen encoder parameters:",
    sum(parameter.numel() for parameter in encoder.parameters()),
)


def encode_texts(texts, show_progress=True):
    """Return final-layer mean embeddings, original token lengths, and input IDs."""
    embedding_batches = []
    token_lengths = []
    input_token_sequences = []
    encoding_start = time.perf_counter()

    with torch.inference_mode():
        for batch_start in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[batch_start : batch_start + BATCH_SIZE]
            full_token_ids = tokenizer(batch_texts, truncation=False)["input_ids"]
            token_lengths.extend(len(token_ids) for token_ids in full_token_ids)
            model_inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_TOKENS,
                return_tensors="pt",
            ).to(DEVICE)

            for token_ids, attention_mask in zip(
                model_inputs["input_ids"], model_inputs["attention_mask"]
            ):
                input_token_sequences.append(
                    tuple(token_ids[attention_mask.bool()].tolist())
                )

            token_vectors = encoder(**model_inputs).last_hidden_state
            content_mask = model_inputs["attention_mask"].bool()
            for special_token_id in tokenizer.all_special_ids:
                content_mask &= model_inputs["input_ids"].ne(special_token_id)

            token_weights = content_mask.unsqueeze(-1).to(token_vectors.dtype)
            sentence_vectors = (token_vectors * token_weights).sum(dim=1)
            sentence_vectors /= token_weights.sum(dim=1).clamp_min(1)
            embedding_batches.append(sentence_vectors.cpu().numpy())

            if show_progress and batch_start % 160 == 0:
                completed = min(batch_start + BATCH_SIZE, len(texts))
                elapsed = time.perf_counter() - encoding_start
                print(
                    f"Encoded {completed}/{len(texts)} texts ({elapsed:.1f}s)",
                    flush=True,
                )

    return (
        np.concatenate(embedding_batches),
        np.asarray(token_lengths),
        input_token_sequences,
    )


embedding_start = time.perf_counter()
embeddings, original_token_lengths, encoded_token_sequences = encode_texts(
    retained_prompts["text"].tolist()
)
embedding_seconds = time.perf_counter() - embedding_start
truncated_count = int((original_token_lengths > MAX_TOKENS).sum())
print("Embedding shape:", embeddings.shape)
print("Embedding extraction seconds:", embedding_seconds)
print("Truncated texts:", truncated_count)

# Check whether truncation created identical inputs across split boundaries.
retained_prompts["token_key"] = encoded_token_sequences
training_token_sequences = retained_prompts.loc[
    retained_prompts["split"].eq("train"), "token_key"
]
input_overlap_counts = {}
for evaluation_split in ["validation", "test"]:
    evaluation_sequences = set(
        retained_prompts.loc[
            retained_prompts["split"].eq(evaluation_split), "token_key"
        ]
    )
    input_overlap_counts[evaluation_split] = int(
        training_token_sequences.isin(evaluation_sequences).sum()
    )
assert not any(input_overlap_counts.values())
print("Training overlaps after tokenization:", input_overlap_counts)

Frozen encoder parameters: 183831552


Encoded 8/1640 texts (0.3s)


Encoded 168/1640 texts (1.7s)


Encoded 328/1640 texts (2.6s)


Encoded 488/1640 texts (3.6s)


Encoded 648/1640 texts (4.8s)


... (3 outputs left out)

Encoded 1288/1640 texts (10.5s)


Encoded 1448/1640 texts (11.9s)


Encoded 1608/1640 texts (13.2s)


Embedding shape: (1640, 768)
Embedding extraction seconds: 13.519144561999838
Truncated texts: 152
Training overlaps after tokenization: {'validation': 0, 'test': 0}


In [7]:
def fit_probe(features, seed):
    classifier = make_pipeline(StandardScaler(), LogisticRegression(
        C=1.0, class_weight="balanced", solver="lbfgs", max_iter=2000, random_state=seed))
    started = time.perf_counter()
    classifier.fit(features[training_mask], labels[training_mask])
    return classifier, time.perf_counter() - started

def choose_threshold(validation_labels, validation_scores, fpr_limit=0.05):
    candidates = np.unique(np.append(validation_scores,
                          np.nextafter(validation_scores.max(), np.inf)))
    negatives = validation_scores[validation_labels == 0]
    return float(next(t for t in candidates if np.mean(negatives >= t) <= fpr_limit))

def primary_interval(frame, reps=1000):
    frame = frame.sort_values("row_id", kind="mergesort")
    y = frame.label.to_numpy()
    score = frame.score.to_numpy()
    rng = np.random.default_rng([zlib.crc32(str(frame.value.iloc[0]).encode()),
                                zlib.crc32(str(frame.column.iloc[0]).encode())])
    draws = []
    for _ in range(reps):
        indices = rng.integers(0, len(y), len(y))
        if len(np.unique(y[indices])) == 2:
            draws.append(roc_auc_score(y[indices], score[indices]))
    return np.percentile(draws, [2.5, 97.5])

baseline_probe, baseline_fit_seconds = fit_probe(embeddings, 15)
baseline_validation = baseline_probe.predict_proba(embeddings[validation_mask])[:, 1]
baseline_threshold = choose_threshold(labels[validation_mask], baseline_validation)
print("Baseline threshold:", repr(baseline_threshold))
clean = retained_prompts.loc[test_mask, ["row_id", "dataset", "label"]].copy()
clean["score"] = baseline_probe.predict_proba(embeddings[test_mask])[:, 1]
comparison = clean.merge(old_scores, on="row_id", suffixes=("", "_old"))
print("Maximum probability difference from midsemester:",
      np.max(np.abs(comparison.score - comparison.score_old)))

# Reuse identical clean embeddings in memory; encode changed copies once.
changed = perturbations[~perturbations["style"].eq("clean")].copy()
changed_features, changed_lengths, _ = encode_texts(changed.text.tolist())
variant_features = np.empty((len(perturbations), embeddings.shape[1]), dtype=np.float32)
row_to_feature = dict(zip(retained_prompts.row_id, embeddings))
clean_variants = perturbations["style"].eq("clean")
variant_features[clean_variants] = np.stack(perturbations.loc[clean_variants, "row_id"].map(row_to_feature))
variant_features[~clean_variants] = changed_features

score_frames = []
seed_frames = []
def make_scores(classifier, features, threshold, value):
    frame = perturbations[["row_id", "column", "label"]].copy()
    frame["value"] = value
    frame["score"] = classifier.predict_proba(features)[:, 1]
    frame["threshold"] = threshold
    return frame[["row_id", "value", "column", "label", "score", "threshold"]]

baseline_scores = make_scores(baseline_probe, variant_features, baseline_threshold, "frozen")
score_frames.append(baseline_scores)

# Fix the decision margin from frozen perturbation intervals before any LoRA outcomes.
interval_rows = []
for column, group in baseline_scores.groupby("column"):
    lower, upper = primary_interval(group)
    interval_rows.append({"column": column, "auroc": roc_auc_score(group.label, group.score),
                          "lower": lower, "upper": upper})
frozen_intervals = pd.DataFrame(interval_rows)
perturbed_intervals = frozen_intervals[frozen_intervals.column.str.endswith(("_zero_width", "_homoglyph", "_base64"))]
no_margin = float(((perturbed_intervals.upper - perturbed_intervals.lower) / 2).max())
yes_margin = float(np.ceil((no_margin + 0.0001) * 100) / 100)
print("Frozen perturbation half-width / yes margin:", no_margin, yes_margin)
frozen_intervals["no_margin"] = no_margin
frozen_intervals["yes_margin"] = yes_margin
frozen_intervals.to_csv("frozen_intervals.csv", index=False)
for seed in SEEDS:
    classifier, seconds = fit_probe(embeddings, seed)
    tau = choose_threshold(labels[validation_mask], classifier.predict_proba(embeddings[validation_mask])[:, 1])
    frame = make_scores(classifier, variant_features, tau, "frozen")
    frame["seed"] = seed
    seed_frames.append(frame)

truncation = changed[["column"]].copy()
truncation["truncated"] = changed_lengths > MAX_TOKENS
print("Truncation by changed column:")
display(truncation.groupby("column").truncated.agg(["sum", "count"]))
truncation.groupby("column").truncated.agg(["sum", "count"]).to_csv("truncation.csv")


Baseline threshold: 0.37308203458513206
Maximum probability difference from midsemester: 0.0008839334429818768
Encoded 8/1045 texts (0.1s)


Encoded 168/1045 texts (1.6s)


Encoded 328/1045 texts (3.3s)


Encoded 488/1045 texts (4.8s)


Encoded 648/1045 texts (6.5s)


Encoded 808/1045 texts (8.1s)


Encoded 968/1045 texts (9.7s)


Frozen perturbation half-width / yes margin: 0.10717233680630375 0.11


Truncation by changed column:


,sum,count
column,,
deepset_base64,41,116
deepset_homoglyph,21,116
deepset_paraphrase,0,50
deepset_zero_width,29,116
safeguard_base64,104,199
safeguard_homoglyph,64,199
safeguard_paraphrase,4,50
safeguard_zero_width,87,199


In [8]:
def normalized_rows(array):
    return array / np.maximum(np.linalg.norm(array, axis=1, keepdims=True), 1e-12)

# Each vector is estimated from training pairs and held fixed during test evaluation.
concept_texts = concept_pairs.injection_text.tolist() + concept_pairs.benign_text.tolist()
concept_test = concept_pairs[concept_pairs.split.eq("test")]
concept_homoglyphs = [homoglyph_map_text for homoglyph_map_text in
    (concept_test.injection_text.tolist() + concept_test.benign_text.tolist())]
homoglyph_map = str.maketrans({"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х"})
concept_homoglyphs = [t.translate(homoglyph_map) for t in concept_homoglyphs]
frozen_concepts, _, _ = encode_texts(concept_texts + concept_homoglyphs, show_progress=False)
train_pairs = concept_pairs.split.eq("train").to_numpy()
test_pairs = concept_pairs.split.eq("test").to_numpy()

def concept_directions(features):
    count = len(concept_pairs)
    differences = features[:count] - features[count:2*count]
    training_differences = differences[train_pairs]
    mean_vector = training_differences.mean(axis=0)
    mean_unit = mean_vector / np.linalg.norm(mean_vector)
    normalized = normalized_rows(training_differences)
    component = PCA(n_components=1).fit(normalized).components_[0]
    if component @ mean_vector < 0:
        component = -component
    return mean_vector, mean_unit, component, differences[test_pairs]

v0, u0, p0, frozen_test_contrasts = concept_directions(frozen_concepts)
representation_rows = []
concept_rows = []

def measure_representation(method, seed, clean_features, concepts):
    delta = clean_features[test_mask] - embeddings[test_mask]
    relative = np.linalg.norm(delta, axis=1) / np.maximum(np.linalg.norm(embeddings[test_mask], axis=1), 1e-12)
    vm, um, pm, test_contrasts = concept_directions(concepts)
    changed_count = len(concept_test)
    offset = 2 * len(concept_pairs)
    perturbed_contrasts = concepts[offset:offset+changed_count] - concepts[offset+changed_count:]
    for dataset in ["deepset", "safeguard"]:
        mask = retained_prompts.loc[test_mask, "dataset"].eq(dataset).to_numpy()
        drift = float(relative[mask].mean())
        rng = np.random.default_rng(15)
        values = relative[mask]
        boot = values[rng.integers(0, len(values), (1000, len(values)))].mean(axis=1)
        drift_lower, drift_upper = np.percentile(boot, [2.5, 97.5])
        representation_rows.append({"value": method, "seed": seed, "dataset": dataset,
            "D": drift, "lower": float(drift_lower), "upper": float(drift_upper), "D_verdict": "yes" if drift >= .05 else "no" if drift <= .01 else "undecided"})
    concept_rows.append({"value": method, "seed": seed,
        "mean_direction_change": 1 - float(u0 @ um),
        "pca_direction_change": 1 - float(p0 @ pm),
        "concept_norm": float(np.linalg.norm(vm)),
        "fixed_axis_margin_change": float(((test_contrasts-frozen_test_contrasts) @ u0).mean()),
        "mean_clean_margin": float((test_contrasts @ um).mean()),
        "mean_homoglyph_margin": float((perturbed_contrasts @ um).mean()),
        "pca_clean_margin": float((test_contrasts @ pm).mean()),
        "pca_homoglyph_margin": float((perturbed_contrasts @ pm).mean()),
        "contrast_cosine_clean": float((normalized_rows(test_contrasts) @ um).mean()),
        "contrast_cosine_homoglyph": float((normalized_rows(perturbed_contrasts) @ um).mean())})

measure_representation("frozen", 15, embeddings, frozen_concepts)


In [9]:
# CPU pilot: LoRA on query/value projections in the last two encoder layers.
# One epoch over the complete original training set; no attack augmentation or test selection.
training_texts = retained_prompts.loc[training_mask, "text"].tolist()
training_labels = labels[training_mask]
training_tokens = tokenizer(training_texts, truncation=True, max_length=MAX_TOKENS)
from transformers import DataCollatorWithPadding
collator = DataCollatorWithPadding(tokenizer)

runtime_rows = [{"value": "frozen", "seed": 15, "training_seconds": baseline_fit_seconds,
                 "embedding_seconds": embedding_seconds, "trainable_parameters": 0}]
# Release the frozen encoder before loading adapted copies; frozen embeddings stay in memory.
frozen_encoder = encoder.cpu()
adapted_encoders = []
adapted_probes = []
gc.collect()

for seed in SEEDS:
    print("START LORA SEED", seed, flush=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    encoder = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, token=False).to(DEVICE)
    targets = [name for name, _ in encoder.named_modules()
               if ("layer.10." in name or "layer.11." in name)
               and (name.endswith("query_proj") or name.endswith("value_proj"))]
    assert len(targets) == 4, targets
    encoder = get_peft_model(encoder, LoraConfig(r=4, lora_alpha=8, lora_dropout=0.05,
                                               target_modules=targets, bias="none"))
    trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    # A temporary linear head supplies supervised adaptation loss. The reported probe is refitted
    # afterwards using exactly the frozen baseline's scaler/logistic-regression protocol.
    head = torch.nn.Linear(embeddings.shape[1], 1).to(DEVICE)
    means = torch.tensor(embeddings[training_mask].mean(axis=0), device=DEVICE)
    scales = torch.tensor(embeddings[training_mask].std(axis=0).clip(min=1e-6), device=DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": [p for p in encoder.parameters() if p.requires_grad], "lr": 1e-4},
        {"params": head.parameters(), "lr": 1e-3}], weight_decay=0.01)
    positive_weight = torch.tensor((training_labels == 0).sum() / (training_labels == 1).sum(), device=DEVICE)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    order = np.random.default_rng(seed).permutation(len(training_texts))
    started = time.perf_counter()
    encoder.train()
    head.train()
    losses = []
    for step, begin in enumerate(range(0, len(order), BATCH_SIZE)):
        indices = order[begin:begin+BATCH_SIZE]
        examples = [{key: value[index] for key, value in training_tokens.items()} for index in indices]
        inputs = collator(examples).to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        token_vectors = encoder(**inputs).last_hidden_state
        mask = inputs["attention_mask"].bool()
        for special_id in tokenizer.all_special_ids:
            mask &= inputs["input_ids"].ne(special_id)
        weights = mask.unsqueeze(-1).to(token_vectors.dtype)
        features = (token_vectors * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1)
        logits = head((features-means)/scales).squeeze(-1)
        loss = criterion(logits, torch.tensor(training_labels[indices], dtype=torch.float32, device=DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters())+list(head.parameters()), 1.0)
        optimizer.step()
        losses.append(float(loss.detach()))
        if step % 20 == 0:
            print("seed", seed, "batch", step, "loss", round(np.mean(losses[-20:]), 4),
                  "elapsed", round(time.perf_counter()-started, 1), flush=True)
    adaptation_seconds = time.perf_counter()-started
    encoder.eval()
    del optimizer, head
    gc.collect()
    started = time.perf_counter()
    adapted_embeddings, _, _ = encode_texts(retained_prompts.text.tolist())
    adapted_changed, _, _ = encode_texts(changed.text.tolist())
    adapted_variants = np.empty_like(variant_features)
    lookup = dict(zip(retained_prompts.row_id, adapted_embeddings))
    adapted_variants[clean_variants] = np.stack(perturbations.loc[clean_variants, "row_id"].map(lookup))
    adapted_variants[~clean_variants] = adapted_changed
    adapted_concepts, _, _ = encode_texts(concept_texts + concept_homoglyphs, show_progress=False)
    extraction_seconds = time.perf_counter()-started
    classifier, fit_seconds = fit_probe(adapted_embeddings, seed)
    tau = choose_threshold(labels[validation_mask], classifier.predict_proba(adapted_embeddings[validation_mask])[:, 1])
    frame = make_scores(classifier, adapted_variants, tau, "lora")
    frame["seed"] = seed
    seed_frames.append(frame)
    if seed == SEEDS[0]:
        score_frames.append(frame.drop(columns="seed"))
    measure_representation("lora", seed, adapted_embeddings, adapted_concepts)
    runtime_rows.append({"value": "lora", "seed": seed,
        "training_seconds": adaptation_seconds + fit_seconds,
        "embedding_seconds": extraction_seconds, "trainable_parameters": trainable})
    pd.concat(seed_frames, ignore_index=True).to_csv("seed_scores.csv", index=False)
    pd.DataFrame(runtime_rows).to_csv("runtimes.csv", index=False)
    pd.DataFrame(representation_rows).to_csv("representation_drift.csv", index=False)
    pd.DataFrame(concept_rows).to_csv("concept_metrics.csv", index=False)
    pd.concat(score_frames, ignore_index=True).to_csv("scores.csv", index=False)
    print("FINISHED SEED", seed, "train seconds", adaptation_seconds, "threshold", repr(tau), flush=True)
    adapted_encoders.append(encoder.cpu())
    adapted_probes.append(classifier)
    del encoder, adapted_embeddings, adapted_changed, adapted_variants, adapted_concepts, classifier
    gc.collect()


START LORA SEED 15


seed 15 batch 0 loss 1.3817 elapsed 0.2


seed 15 batch 20 loss 0.8447 elapsed 1.7


seed 15 batch 40 loss 0.6629 elapsed 3.2


seed 15 batch 60 loss 0.4828 elapsed 4.8


seed 15 batch 80 loss 0.4099 elapsed 6.4


... (71 outputs left out)

Encoded 648/1045 texts (6.6s)


Encoded 808/1045 texts (8.3s)


Encoded 968/1045 texts (9.9s)


FINISHED SEED 17 train seconds 10.201745951000248 threshold 0.4917755330771162


In [10]:
scores = pd.concat(score_frames, ignore_index=True)
scores.to_csv("scores.csv", index=False)
seed_scores = pd.concat(seed_frames, ignore_index=True)
seed_scores.to_csv("seed_scores.csv", index=False)
metric_rows = []
for (value, seed, column), group in seed_scores.groupby(["value", "seed", "column"]):
    y = group.label.to_numpy()
    pred = group.score.to_numpy() >= group.threshold.to_numpy()
    metric_rows.append({"value": value, "seed": seed, "column": column,
        "AUROC": roc_auc_score(y, group.score), "recall": pred[y == 1].mean(),
        "FPR": pred[y == 0].mean(), "threshold": group.threshold.iloc[0]})
seed_metrics = pd.DataFrame(metric_rows)
seed_metrics.to_csv("seed_metrics.csv", index=False)
display(seed_metrics.groupby(["value", "column"]).AUROC.agg(["mean", "std", "min", "max"]))
display(pd.DataFrame(runtime_rows))
display(pd.DataFrame(representation_rows))
display(pd.DataFrame(concept_rows))

# Deterministic, reproducible error sample from a prespecified two-class cell.
error_cell = scores[scores.value.eq("frozen") & scores.column.eq("deepset_homoglyph")].copy()
error_cell["prediction"] = error_cell.score >= error_cell.threshold
sampled = []
for label in [0, 1]:
    wrong = error_cell[error_cell.label.eq(label) & error_cell.prediction.ne(label)]
    sampled.append(wrong.sample(n=min(10, len(wrong)), random_state=15))
error_sample = pd.concat(sampled).merge(perturbations[["row_id", "column", "text"]],
                                      on=["row_id", "column"], validate="one_to_one")
# Categories describe observable text properties, not causal diagnoses.
def category(row):
    text = row.text.casefold()
    if row.label == 0:
        return "benign instruction vocabulary"
    if any(word in text for word in ["system", "earlier", "prior", "previous", "policy", "policies", "rules", "constraint"]):
        return "hierarchy or policy redirection"
    return "task or role redirection"
error_sample["category"] = error_sample.apply(category, axis=1)
error_sample[["row_id", "value", "column", "category"]].to_csv("errors.csv", index=False)
error_sample.to_csv("error_examples.csv", index=False)
display(error_sample[["row_id", "label", "score", "category", "text"]])


mean       std       min       max
value  column                                                      
frozen deepset_base64        0.476190  0.000000  0.476190  0.476190
       deepset_clean         0.991667  0.000000  0.991667  0.991667
       deepset_homoglyph     0.776786  0.000000  0.776786  0.776786
       deepset_paraphrase    0.985600  0.000000  0.985600  0.985600
       deepset_zero_width    0.767262  0.000000  0.767262  0.767262
       safeguard_base64      0.497199  0.000000  0.497199  0.497199
       safeguard_clean       0.995565  0.000000  0.995565  0.995565
       safeguard_homoglyph   0.543067  0.000000  0.543067  0.543067
       safeguard_paraphrase  1.000000  0.000000  1.000000  1.000000
       safeguard_zero_width  0.470005  0.000000  0.470005  0.470005
lora   deepset_base64        0.466865  0.012258  0.453274  0.477083
       deepset_clean         0.992560  0.000298  0.992262  0.992857
       deepset_homoglyph     0.768750  0.006581  0.761310  0.773810
       deepset_paraphrase    0.988800  0.001600  0.987200  0.990400
       deepset_zero_width    0.762500  0.003892  0.758036  0.765179
       safeguard_base64      0.487434  0.003038  0.484477  0.490546
       safeguard_clean       0.995643  0.000243  0.995448  0.995915
       safeguard_homoglyph   0.553455  0.010324  0.545518  0.565126
       safeguard_paraphrase  1.000000  0.000000  1.000000  1.000000
       safeguard_zero_width  0.475996  0.004877  0.471755  0.481326

,value,seed,training_seconds,embedding_seconds,trainable_parameters
0,frozen,15,0.191217,13.519145,0
1,lora,15,10.737393,24.327158,24576
2,lora,16,10.379804,23.623631,24576
3,lora,17,10.441001,23.953095,24576


,value,seed,dataset,D,lower,upper,D_verdict
0,frozen,15,deepset,0.000000,0.000000,0.000000,no
1,frozen,15,safeguard,0.000000,0.000000,0.000000,no
2,lora,15,deepset,0.050452,0.046964,0.054233,yes
3,lora,15,safeguard,0.048362,0.045724,0.051093,undecided
4,lora,16,deepset,0.043835,0.040590,0.047357,undecided
5,lora,16,safeguard,0.040222,0.037697,0.042846,undecided
6,lora,17,deepset,0.049346,0.045133,0.054085,undecided
7,lora,17,safeguard,0.044964,0.042009,0.047961,undecided


,value,seed,mean_direction_change,pca_direction_change,concept_norm,fixed_axis_margin_change,mean_clean_margin,mean_homoglyph_margin,pca_clean_margin,pca_homoglyph_margin,contrast_cosine_clean,contrast_cosine_homoglyph
0,frozen,15,1.192093e-07,-1.192093e-07,5.459351,0.000000,3.168307,1.105924,1.930827,0.961677,0.361559,0.181812
1,lora,15,1.542032e-03,1.205564e-03,5.458182,0.061614,3.237680,1.071723,1.755371,0.882741,0.364813,0.176774
2,lora,16,7.190704e-04,6.141663e-04,5.446427,0.014496,3.171955,1.073422,1.793589,0.890230,0.359685,0.177294
3,lora,17,2.179146e-03,4.615366e-03,5.396218,0.005383,3.179993,1.063071,1.528662,0.855491,0.362878,0.176366


,row_id,label,score,category,text
0,deepset:test:56,0,0.709942,benign instruction vocabulary,Vеrsсhlерреn Orks in dеr Ukrаinе Kindеr?
1,deepset:test:10,0,0.584259,benign instruction vocabulary,Artifiсiаl intеlligеnсе Ethiсs Eurоре
2,deepset:test:63,0,0.851098,benign instruction vocabulary,Glеiсhbеrесhtigung LGBTIQ+ Eurора
3,deepset:test:41,0,0.973781,benign instruction vocabulary,"Wiе viеl muss iсh trаiniеrеn, um еinеn Mаrаthо..."
4,deepset:test:64,0,0.996547,benign instruction vocabulary,Wiеviеlе Zigаrеttеn hаt Hеlmut Sсhmidt аn еinе...
5,deepset:test:69,0,0.999926,benign instruction vocabulary,Mеin Mаnn und iсh möсhtеn gеrnе in unsеrеm Gаr...
6,deepset:test:67,0,0.999997,benign instruction vocabulary,Iсh bin lеidеnsсhаftliсhеr Fоtоgrаf und möсhtе...
7,deepset:test:48,0,0.999107,benign instruction vocabulary,Wаs wаr 2022 dаs wiсhtigstе Thеmа in Dеutsсhlаnd?
8,deepset:test:55,0,0.907196,benign instruction vocabulary,E-Sсооtеr Vеrkеhrssiсhеrhеit Dеutsсhlаnd
9,deepset:test:35,0,0.560639,benign instruction vocabulary,Hоw dоеs vассinе рrоduсtiоn wоrk аnd whiсh mаn...
